# Train CNN + Fusion on Colab GPU

**Before you run:** Runtime → Change runtime type → **GPU** (T4 is fine).

What this notebook does:
1. Mount Google Drive (where `frames/` and `pose/` already live).
2. Clone the repo from GitHub.
3. Train the **CNN encoder** with a player-identity auxiliary objective (the original over/under target was random noise — see PR notes).
4. Train the **Fusion model** end-to-end (LSTM + new CNN encoder).
5. Evaluate on the test set.
6. Copy checkpoints back to Drive so they survive runtime resets.

Expected wall-clock on a T4: a few minutes for CNN, a few minutes for fusion.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
candidates = [
    '/content/drive/MyDrive/podz-liability-fund',
    '/content/drive/MyDrive/Podz-Liability-Fund',
    '/content/drive/Shareddrives/podz-liability-fund',
]
GDRIVE = next((p for p in candidates if os.path.isdir(p)), None)
if GDRIVE is None:
    raise FileNotFoundError('Drive folder podz-liability-fund not found. Add a shortcut to My Drive.')
os.environ['PODZ_DATA_DIR'] = GDRIVE
print('Using:', GDRIVE)
print('Contents:', sorted(os.listdir(GDRIVE)))

## 2. Clone repo + install deps

In [ ]:
import os, sys, subprocess
REPO_DIR = '/content/Podz-Liability-Fund'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Nathan-Dela-Pena/Podz-Liability-Fund.git',
                    REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('repo:', REPO_DIR)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Runtime → Change runtime type → GPU')

## 3. Sanity check data

`config.py` reads `PODZ_DATA_DIR` from the env, so frames/pose come straight from Drive.

In [ ]:
from config import FRAMES_DIR, POSE_DIR, PLAYERS
import pathlib, glob
print('FRAMES_DIR =', FRAMES_DIR)
print('POSE_DIR   =', POSE_DIR)
print('Frames JPGs:', len(glob.glob(f'{FRAMES_DIR}/**/*.jpg', recursive=True)))
print('Pose CSVs: ', len(list(pathlib.Path(POSE_DIR).glob('*.csv'))))
print('Players:   ', len(PLAYERS))

## 4. Train CNN (player-identity encoder)

The CNN no longer tries to predict over/under from a single random clip — there's no signal there. Instead it learns to identify *which player* is in each clip, which produces a player-kinetic-fingerprint embedding. The fusion model consumes that embedding alongside the LSTM stat history.

In [ ]:
from src.models.cnn import train_cnn
cnn_model = train_cnn(device='cuda')

## 5. Train Fusion (over/under)

Loads `checkpoints/lstm_best.pt` (already in the repo) + the freshly trained CNN encoder, then fine-tunes end-to-end.

In [ ]:
from src.models.fusion import train_fusion
fusion_model = train_fusion(device='cuda')

## 6. Evaluate on the test set

In [ ]:
!python scripts/evaluate.py

## 7. Save checkpoints to Drive

Colab wipes `/content` on disconnect — copy the `.pt` files to Drive so you can pull them down to your laptop.

In [ ]:
import shutil, pathlib
src = pathlib.Path('checkpoints')
dst = pathlib.Path(os.environ['PODZ_DATA_DIR']) / 'checkpoints'
dst.mkdir(parents=True, exist_ok=True)
for f in sorted(src.glob('*.pt')):
    shutil.copy2(f, dst / f.name)
    print(f'copied {f.name} ({f.stat().st_size / 1e6:.1f} MB)  →  {dst / f.name}')